In [1]:
from transformers import BartTokenizer
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
from Utils import TimexNorm_Utils
from Reader import obtain_combined_dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = TimexNorm_Utils(tokenizer)

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
tokenizer.add_special_tokens({"additional_special_tokens": ["<timex","type=DATE>","type=TIME>","type=DURATION>","type=SET>","</timex>", "<sep>"]})
model.resize_token_embeddings(len(tokenizer))

Embedding(50272, 1024)

In [3]:
datasets = obtain_combined_dataset(["TempEval3","wikiwars","tweets"], "normalised")

In [4]:
datasets = utils.tokenize_datasets(datasets)

In [5]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results/TimeNormBart",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    predict_with_generate=True,
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    data_collator=data_collator,
    compute_metrics=utils.compute_metrics,
)

In [6]:
trainer.train()

d:\GeoTKG\venv\Lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 15843
  Num Epochs = 1
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 991
  Number of trainable parameters = 406298624


  0%|          | 0/991 [00:00<?, ?it/s]

***** Running Evaluation *****
  Num examples = 1822
  Batch size = 32


{'loss': 0.8937, 'learning_rate': 2.477295660948537e-05, 'epoch': 0.5}


  0%|          | 0/57 [00:00<?, ?it/s]

Saving model checkpoint to ./results/TimeNormBart\checkpoint-500
Configuration saved in ./results/TimeNormBart\checkpoint-500\config.json


{'eval_loss': 0.5117689371109009, 'eval_accuracy strict': 0.5461031833150384, 'eval_accuracy relaxed': 0.5718990120746432, 'eval_runtime': 621.8965, 'eval_samples_per_second': 2.93, 'eval_steps_per_second': 0.092, 'epoch': 0.5}


Model weights saved in ./results/TimeNormBart\checkpoint-500\pytorch_model.bin


Training completed. Do not forget to share your model on huggingface.co/models =)




{'train_runtime': 12890.786, 'train_samples_per_second': 1.229, 'train_steps_per_second': 0.077, 'train_loss': 0.5238498645401386, 'epoch': 1.0}


TrainOutput(global_step=991, training_loss=0.5238498645401386, metrics={'train_runtime': 12890.786, 'train_samples_per_second': 1.229, 'train_steps_per_second': 0.077, 'train_loss': 0.5238498645401386, 'epoch': 1.0})

In [9]:
trainer.save_model("./results/TimeNormBart")
tokenizer.save_pretrained("./results/TimeNormBart")

Saving model checkpoint to ./results/TimeNormBart
Configuration saved in ./results/TimeNormBart\config.json
Model weights saved in ./results/TimeNormBart\pytorch_model.bin
tokenizer config file saved in ./results/TimeNormBart\tokenizer_config.json
Special tokens file saved in ./results/TimeNormBart\special_tokens_map.json
added tokens file saved in ./results/TimeNormBart\added_tokens.json


('./results/TimeNormBart\\tokenizer_config.json',
 './results/TimeNormBart\\special_tokens_map.json',
 './results/TimeNormBart\\vocab.json',
 './results/TimeNormBart\\merges.txt',
 './results/TimeNormBart\\added_tokens.json')

In [10]:
trainer.evaluate(datasets["test"])

***** Running Evaluation *****
  Num examples = 739
  Batch size = 32


  0%|          | 0/24 [00:00<?, ?it/s]

{'eval_loss': 0.3016822040081024,
 'eval_accuracy strict': 0.6116373477672531,
 'eval_accuracy relaxed': 0.6238159675236806,
 'eval_runtime': 281.0752,
 'eval_samples_per_second': 2.629,
 'eval_steps_per_second': 0.085,
 'epoch': 1.0}